# Image patching

Use a generated RGB image so this notebook runs without downloading data. The patch extractor only rearranges pixels; normalization happens in the image-loading cell. Choose image dimensions divisible by the patch size.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import torch

from utils.embedding import patchify, PatchEmbeddingModel
from utils.visualization import display_patches

torch.manual_seed(0)
np.random.seed(0)

image_size = 224
patch_size = 16
hidden_dim = 192

# Gradients and a checkerboard make the patches easy to recognize.
rows, cols = np.meshgrid(np.arange(image_size), np.arange(image_size), indexing="ij")
pixels = np.stack((
    cols * 255 // (image_size - 1),
    rows * 255 // (image_size - 1),
    ((rows // patch_size + cols // patch_size) % 2) * 255,
), axis=-1).astype(np.uint8)
image = Image.fromarray(pixels)

# To use your own photo, replace the line above with:
# image = Image.open("data/image_8765cd2c.jpg").convert("RGB").resize((image_size, image_size))
image_tensor = torch.from_numpy(np.array(image)).permute(2, 0, 1).unsqueeze(0).float() / 255.0
results = patchify(image_tensor, patch_size=patch_size)
patches = results["patches"]

print(f"image tensor shape: {image_tensor.shape}")
print(f"patches tensor shape: {patches.shape}")
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
display_patches(patches, patch_size=patch_size)

# Patch projection, classification token, and learned positions

Create the model once and reuse it. It owns the linear projection, the shared classification token, and one learned position vector per token (including [CLS]). The returned patch embeddings are the raw projection; the complete token sequence also includes [CLS] and positions.

For 224 × 224 RGB images and 16 × 16 patches: [B, 3, 224, 224] → [B, 196, 768] → [B, 196, 192] → [B, 197, 192].

In [ ]:
model = PatchEmbeddingModel(
    embedding_dim=results["embedding_dim"],
    hidden_dim=hidden_dim,
    num_patches=patches.size(1),
)
embedded_patches = model(patches)

print(f"projected patches: {embedded_patches['embedded_patches'].shape}")
print(f"encoder input ([CLS] + patches + positions): {embedded_patches['patches_with_cls_token'].shape}")
print(f"shared classification parameter: {model.cls_token.shape}")
print(f"learned positions: {model.positional_encoding.pe.shape}")

# Inspect gradient flow

This toy loss lets us inspect gradients in the embedding parameters before we build attention. It is only an autograd demonstration; the image-classification objective will come later.

In [ ]:
model.zero_grad()
tokens = model(patches)["patches_with_cls_token"]
loss = tokens.square().mean()
loss.backward()

for name, parameter in model.named_parameters():
    print(f"{name}: shape={tuple(parameter.shape)}, gradient norm={parameter.grad.norm().item():.6f}")